# 🎵 Task 1: Raga Bhairav — Unconditioned Melody Generation with LSTM

**Task type**: Symbolic, Unconditioned Generation — learn $p(x)$ over melodic sequences and sample new melodies.

## Introduction

Indian classical music is built on the concept of **ragas** — melodic frameworks that define which notes can be used, how they should be ordered, which notes are most important, and what characteristic phrases give the raga its identity. Unlike Western scales, ragas carry rules about *movement* — certain notes may only appear in ascent, others only in descent, and specific phrases (*pakad*) act as signatures.

**Raga Bhairav** is one of the oldest and most revered ragas in the Hindustani tradition:
- **Time**: Performed at dawn
- **Mood (rasa)**: Serious, devotional, profound
- **Scale**: Sa, komal Re (♭2), Ga, Ma, Pa, komal Dha (♭6), Ni — the flat 2nd and flat 6th give it a uniquely ancient, solemn sound
- **Vadi** (most important note): Ma (F)
- **Samvadi** (second most important): Sa (C)

**Our goal**: Train an LSTM to learn the melodic distribution of Bhairav and generate new melodies that are musically valid — i.e., the model should independently discover which notes matter, how to move between them, and what patterns define the raga.

### Pipeline Overview
1. Encode Bhairav grammar → generate synthetic training data
2. Tokenize as (pitch, duration) pairs
3. Train a 2-layer LSTM (next-token prediction)
4. Evaluate against baselines (random, Markov chain)
5. Analyze whether the model learned Bhairav's grammar
6. Export generated melodies as MIDI → MP3

## 0. Install Dependencies

In [ ]:
!pip install music21 -q
!apt-get install -y fluidsynth fluid-soundfont-gm ffmpeg -q
!pip install pyfluidsynth -q
print('Done.')

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from music21 import stream, note, instrument, tempo
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import subprocess, os

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

---
# Section 1: Exploratory Analysis, Data Collection & Pre-processing

## 1.1 Data Source & Motivation

**The dataset problem**: Unlike Western classical or pop music, there is no large-scale, publicly available MIDI dataset for Indian classical music. The closest option — a Figshare Hindu Raga MIDI collection — contains only ~20 files, far too few to train any neural model. The CompMusic/Saraga dataset (MTG Barcelona) contains audio recordings with pitch annotations, but extracting clean symbolic data from audio introduces noise and additional engineering complexity beyond the scope of this project.

**Our approach: Grammar-based synthetic generation**. Raga Bhairav's melodic rules are well-documented in music theory texts. We encode these rules explicitly and use them to generate training sequences via a constrained random walk. This is a legitimate approach because:
1. Ragas are *defined* by their grammar — unlike genres like jazz or pop, the rules ARE the music
2. The model's task is to learn these rules implicitly from data, without being told them during training
3. Synthetic generation gives us full control over dataset size and balance

## 1.2 Raga Bhairav Grammar Encoding

In [ ]:
BHAIRAV = {
    'name': 'Bhairav',
    'arohana':   [60, 61, 64, 65, 67, 68, 71, 72],     # S r G m P d N S'
    'avarohana': [72, 71, 68, 67, 65, 64, 61, 60],     # S' N d P m G r S
    'notes': [
        48, 49, 52, 53, 55, 56, 59,   # octave 3
        60, 61, 64, 65, 67, 68, 71,   # octave 4
        72, 73, 76, 77, 79, 80, 83,   # octave 5
    ],
    'vadi':    [53, 65, 77],   # Ma across octaves
    'samvadi': [48, 60, 72],   # Sa across octaves
    'pakad': [
        [61, 60, 68, 67, 65],              # Re Sa Dha Pa Ma
        [64, 65, 67, 68, 67, 65],          # Ga Ma Pa Dha Pa Ma
        [71, 72, 71, 68, 67],              # Ni Sa' Ni Dha Pa
        [60, 61, 64, 65, 64, 61, 60],      # Sa Re Ga Ma Ga Re Sa
        [65, 67, 68, 71, 72],              # Ma Pa Dha Ni Sa'
        [72, 71, 68, 67, 65, 64, 61, 60],  # full avarohana
    ],
    'gamaka': [
        [65, 64, 65], [67, 68, 67],
        [61, 60, 61], [71, 72, 71],
    ]
}

DURATIONS   = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
DUR_WEIGHTS = [0.15, 0.35, 0.10, 0.25, 0.10, 0.05]

NOTE_NAMES = {0:'Sa(C)', 1:'Re(Db)', 4:'Ga(E)', 5:'Ma(F)',
              7:'Pa(G)', 8:'Dha(Ab)', 11:'Ni(B)'}
BHAIRAV_PCS = {p % 12 for p in BHAIRAV['notes']}

print(f"Raga: {BHAIRAV['name']}")
print(f"Scale degrees: {[NOTE_NAMES[p%12] for p in BHAIRAV['arohana']]}")
print(f"Note pool: {len(set(BHAIRAV['notes']))} pitches across 3 octaves")
print(f"Pakad phrases: {len(BHAIRAV['pakad'])}")
print(f"Gamaka patterns: {len(BHAIRAV['gamaka'])}")

## 1.3 Synthetic Data Generation

Each training sequence is produced by a **grammar-constrained random walk** through Bhairav's note set. The walk is controlled by five rules:

| Rule | Effect | Weight |
|------|--------|--------|
| Stepwise motion | Small intervals (1-2 semitones) preferred | 3.0× |
| Direction bias | Notes in current direction (ascending/descending) preferred | 1.8× |
| Vadi emphasis | Ma (F) preferred | 2.5× |
| Samvadi emphasis | Sa (C) preferred | 1.8× |
| Repetition penalty | Same note twice discouraged | 0.3× |

Additionally:
- Direction flips every 4-10 steps → creates natural melodic arcs
- 8% chance per step: inject a **gamaka** (ornamental oscillation)
- 6% chance per step: inject a **pakad** phrase (raga signature)

In [ ]:
def get_note_weights(current_pitch, raga, direction):
    weights = []
    for p in raga['notes']:
        w = 1.0
        interval = p - current_pitch
        if   abs(interval) <= 2: w *= 3.0
        elif abs(interval) <= 4: w *= 1.5
        else:                    w *= 0.4
        if direction == 'up'   and interval > 0: w *= 1.8
        if direction == 'down' and interval < 0: w *= 1.8
        if p in raga['vadi']:    w *= 2.5
        if p in raga['samvadi']: w *= 1.8
        if p == current_pitch:   w *= 0.3
        weights.append(w)
    return weights


def generate_sequence(raga, length=64):
    sequence = []
    scale = raga['notes']
    if random.random() < 0.4:
        pakad = random.choice(raga['pakad'])
        for p in pakad:
            sequence.append((p, random.choices(DURATIONS, DUR_WEIGHTS)[0]))
        current = pakad[-1]
    else:
        current = random.choice(raga['samvadi'])
        sequence.append((current, random.choices(DURATIONS, DUR_WEIGHTS)[0]))
    direction, dir_steps = random.choice(['up','down']), 0

    while len(sequence) < length:
        if random.random() < 0.08:
            for p in random.choice(raga['gamaka']):
                if p in scale: sequence.append((p, 0.25))
            current = sequence[-1][0]; continue
        if random.random() < 0.06:
            for p in random.choice(raga['pakad']):
                sequence.append((p, random.choices(DURATIONS, DUR_WEIGHTS)[0]))
            current = sequence[-1][0]; continue
        dir_steps += 1
        if dir_steps > random.randint(4, 10):
            direction = 'down' if direction == 'up' else 'up'; dir_steps = 0
        weights = get_note_weights(current, BHAIRAV, direction)
        total = sum(weights)
        current = random.choices(scale, [w/total for w in weights])[0]
        sequence.append((current, random.choices(DURATIONS, DUR_WEIGHTS)[0]))
    return sequence[:length]


NUM_SEQUENCES = 5000
SEQ_LENGTH    = 64
print(f'Generating {NUM_SEQUENCES} sequences of length {SEQ_LENGTH}...')
all_sequences = [generate_sequence(BHAIRAV, SEQ_LENGTH) for _ in range(NUM_SEQUENCES)]
print(f'Done. Total tokens: {NUM_SEQUENCES * SEQ_LENGTH:,}')

## 1.4 Exploratory Data Analysis

Before training, we verify the synthetic data is musically valid and understand its statistical properties.

In [ ]:
# Collect all pitches and durations from training data
all_pitches   = [p for seq in all_sequences for p, d in seq]
all_durs      = [d for seq in all_sequences for p, d in seq]
all_pcs       = [p % 12 for p in all_pitches]
all_intervals = []
for seq in all_sequences:
    pitches = [p for p, d in seq]
    all_intervals.extend([pitches[i+1] - pitches[i] for i in range(len(pitches)-1)])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Pitch class distribution
ax = axes[0, 0]
pc_counts = Counter(all_pcs)
pcs_sorted = sorted(BHAIRAV_PCS)
names  = [NOTE_NAMES[pc] for pc in pcs_sorted]
counts = [pc_counts.get(pc, 0) for pc in pcs_sorted]
colors = ['#e74c3c' if pc == 5 else '#3498db' if pc == 0 else '#95a5a6' for pc in pcs_sorted]
ax.bar(names, counts, color=colors)
ax.set_title('Pitch Class Distribution (Red=Vadi, Blue=Samvadi)')
ax.set_ylabel('Count')

# 2. Duration distribution
ax = axes[0, 1]
dur_counts = Counter(all_durs)
durs_sorted = sorted(dur_counts.keys())
ax.bar([str(d) for d in durs_sorted], [dur_counts[d] for d in durs_sorted], color='#2ecc71')
ax.set_title('Duration Distribution (quarter note units)')
ax.set_ylabel('Count')

# 3. Interval distribution (signed)
ax = axes[1, 0]
int_counts = Counter(all_intervals)
int_range = range(min(all_intervals), max(all_intervals)+1)
ax.bar([str(i) for i in int_range], [int_counts.get(i, 0) for i in int_range], color='#9b59b6')
ax.set_title('Interval Distribution (signed semitones)')
ax.set_xlabel('Interval'); ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# 4. Piano roll of one example sequence
ax = axes[1, 1]
example_seq = all_sequences[0]
t = 0
for pitch, dur in example_seq:
    ax.barh(pitch, dur, left=t, height=0.8, color='#e67e22', alpha=0.8)
    t += dur
ax.set_title('Piano Roll — Example Sequence')
ax.set_xlabel('Time (quarter notes)'); ax.set_ylabel('MIDI Pitch')
ytick_pitches = sorted(set(p for p,d in example_seq))
ax.set_yticks(ytick_pitches)
ax.set_yticklabels([note.Note(p).nameWithOctave for p in ytick_pitches], fontsize=7)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Training Data — Exploratory Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150)
plt.show()

# Summary stats
in_scale = sum(1 for pc in all_pcs if pc in BHAIRAV_PCS)
stepwise = sum(1 for iv in all_intervals if abs(iv) <= 2)
print(f'\n── Training Data Summary ──')
print(f'Total tokens:     {len(all_pitches):,}')
print(f'Scale adherence:  {100*in_scale/len(all_pcs):.1f}%')
print(f'Stepwise motion:  {100*stepwise/len(all_intervals):.1f}%')
print(f'Most common note: {NOTE_NAMES[pc_counts.most_common(1)[0][0]]} (should be Ma/Vadi)')
print(f'Unique tokens:    {len(set(tok for seq in all_sequences for tok in seq))}')

---
# Section 2: Modeling

## 2.1 Problem Formulation

We frame unconditioned melody generation as **next-token prediction** — the same formulation used in language modeling:

- **Input**: sequence of the last $W=32$ tokens: $x_{t-W}, x_{t-W+1}, \ldots, x_{t-1}$
- **Output**: probability distribution over the vocabulary for the next token $x_t$
- **Loss**: cross-entropy between predicted distribution and ground truth
- **Vocabulary**: each token is a (pitch, duration) pair — e.g., (F4, 0.5) is one token

At generation time, we sample autoregressively — feed the output back as input to produce arbitrarily long melodies.

## 2.2 Why LSTM?

| Model | Pros | Cons | Verdict |
|-------|------|------|---------|
| **Markov chain (order-$n$)** | Simple, fast, no training | Only $n$ steps of memory; state space explodes with order | Baseline |
| **RNN (vanilla)** | Handles sequences | Vanishing gradients — can't learn long-range patterns | Outdated |
| **LSTM** | Gating mechanism solves vanishing gradients; captures long-range dependencies (32+ tokens) | Slower than Markov; sequential training | **Our choice** |
| **Transformer** | Parallel training; global attention | Overkill for small vocab (~120 tokens); needs more data and GPU time | Too expensive |

LSTM is the sweet spot — powerful enough to learn raga grammar over 32-token windows, small enough to train on a single GPU in under 30 minutes.

## 2.3 Architecture

In [ ]:
# ── Tokenization ──────────────────────────────────────────────────────────────
all_tokens  = [tok for seq in all_sequences for tok in seq]
vocab       = sorted(set(all_tokens))
token2idx   = {t: i for i, t in enumerate(vocab)}
idx2token   = {i: t for t, i in token2idx.items()}
VOCAB_SIZE  = len(vocab)
indexed_sequences = [[token2idx[t] for t in seq] for seq in all_sequences]

print(f'Vocabulary size: {VOCAB_SIZE} unique (pitch, duration) tokens')
print(f'Example: (F4, 0.5) → index {token2idx.get((65, 0.5), "N/A")}')

In [ ]:
# ── Sliding-window dataset ────────────────────────────────────────────────────
WINDOW_SIZE = 32

class RagaDataset(Dataset):
    def __init__(self, sequences, window_size):
        self.samples = []
        for seq in sequences:
            for i in range(len(seq) - window_size):
                self.samples.append((seq[i:i+window_size], seq[i+window_size]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

split      = int(0.9 * len(indexed_sequences))
train_ds   = RagaDataset(indexed_sequences[:split], WINDOW_SIZE)
val_ds     = RagaDataset(indexed_sequences[split:], WINDOW_SIZE)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds):,} samples | Val: {len(val_ds):,} samples')

In [ ]:
# ── LSTM Model ────────────────────────────────────────────────────────────────
class RagaLSTM(nn.Module):
    """
    Architecture:
      Embedding (vocab → 64-dim) → 2-layer LSTM (128 hidden) → Dropout → Linear (→ vocab)
    
    Design choices:
      - embed_dim=64: sufficient for ~120 token vocab; larger wastes capacity
      - hidden_dim=128: balances expressiveness vs. overfitting on synthetic data
      - 2 layers: captures hierarchical patterns (note-level + phrase-level)
      - dropout=0.5: aggressive regularization since synthetic data has limited diversity
    """
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                 dropout=dropout, batch_first=True)
        self.drop      = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb         = self.drop(self.embedding(x))
        out, hidden = self.lstm(emb, hidden)
        logits      = self.fc(self.drop(out[:, -1, :]))
        return logits, hidden

model = RagaLSTM(VOCAB_SIZE).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

## 2.4 Training

- **Loss**: Cross-entropy (standard for next-token prediction)
- **Optimizer**: Adam (lr=1e-3)
- **Scheduler**: ReduceLROnPlateau — halves LR when val loss stalls for 3 epochs
- **Gradient clipping**: max norm = 1.0 (prevents exploding gradients in LSTMs)
- **Early stopping**: patience = 7 epochs

In [ ]:
EPOCHS   = 50
LR       = 1e-3
PATIENCE = 7

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

train_losses, val_losses = [], []
best_val_loss = float('inf')
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    avg_train = total / len(train_loader)

    model.eval()
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            total += criterion(model(xb)[0], yb).item()
    avg_val = total / len(val_loader)

    scheduler.step(avg_val)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val; no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}'); break

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | LR: {optimizer.param_groups[0]["lr"]:.6f}')

print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label='Train', color='steelblue')
axes[0].plot(val_losses,   label='Val',   color='coral')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Perplexity = exp(loss)
axes[1].plot([np.exp(l) for l in train_losses], label='Train', color='steelblue')
axes[1].plot([np.exp(l) for l in val_losses],   label='Val',   color='coral')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity Curves'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

print(f'Final train perplexity: {np.exp(train_losses[-1]):.2f}')
print(f'Best val perplexity:    {np.exp(best_val_loss):.2f}')

## 2.5 Generation

Autoregressive sampling with **temperature** control:
- Temperature < 1: sharper distribution → more conservative melodies
- Temperature = 1: model's learned distribution as-is
- Temperature > 1: flatter distribution → more creative/risky

In [ ]:
def generate_melody(model, seed_tokens, length=96, temperature=1.0):
    model.eval()
    generated = list(seed_tokens)
    with torch.no_grad():
        for _ in range(length):
            context = generated[-WINDOW_SIZE:]
            x = torch.tensor([context], dtype=torch.long).to(DEVICE)
            logits, _ = model(x)
            probs = torch.softmax(logits[0] / temperature, dim=-1)
            generated.append(torch.multinomial(probs, 1).item())
    return [idx2token[i] for i in generated[len(seed_tokens):]]


model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

# Seed with classic Bhairav pakad: Re Sa Dha Pa Ma
seed_phrase = BHAIRAV['pakad'][0]
seed_tokens = []
for p in seed_phrase:
    key = (p, 0.5)
    if key in token2idx: seed_tokens.append(token2idx[key])
    else: seed_tokens.append(token2idx[min(token2idx, key=lambda t: abs(t[0]-p))])

melodies = {}
for temp in [0.5, 1.0, 1.5]:
    mel = generate_melody(model, seed_tokens, length=96, temperature=temp)
    melodies[temp] = mel
    print(f'Temp {temp}: {len(mel)} tokens generated')

---
# Section 3: Evaluation

## 3.1 Evaluation Framework

Music generation evaluation is inherently multi-dimensional. We use both objective metrics and musical property analysis:

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Perplexity** | How well the model predicts held-out data | Lower = better language model, but doesn't guarantee musicality |
| **Scale adherence** | % of notes within Bhairav's 7-note scale | 100% = never breaks the raga |
| **Vadi ratio** | Is Ma (F) the most frequent note? | Bhairav requires Ma to dominate |
| **Stepwise motion %** | % of intervals ≤ 2 semitones | Indian classical music is predominantly stepwise |
| **Pitch entropy** | Diversity of notes used | Too low = repetitive; too high = random |
| **Interval distribution** | Similarity to training data intervals | Measured via KL divergence |

**Key insight**: low perplexity and high musicality don't always align. A model could have perfect perplexity by memorizing sequences, but generate boring music. Conversely, creative generation may have higher perplexity but sound more interesting. We need both types of evaluation.

## 3.2 Baselines

To demonstrate that the LSTM is actually learning something meaningful, we compare against two baselines:
1. **Random baseline**: uniformly sample notes from Bhairav's scale with random durations
2. **Markov chain (order-1)**: predict next note based only on the current note (transition matrix)

In [ ]:
# ── Baseline 1: Random ────────────────────────────────────────────────────────
def generate_random_baseline(length=96):
    """Uniformly sample from Bhairav scale with random durations."""
    return [(random.choice(BHAIRAV['notes']),
             random.choices(DURATIONS, DUR_WEIGHTS)[0])
            for _ in range(length)]


# ── Baseline 2: Order-1 Markov Chain ───────────────────────────────────────────
class MarkovChain:
    """First-order Markov chain over (pitch, duration) tokens."""
    def __init__(self):
        self.transitions = defaultdict(Counter)

    def fit(self, sequences):
        for seq in sequences:
            for i in range(len(seq) - 1):
                self.transitions[seq[i]][seq[i+1]] += 1
        # Normalize
        self.probs = {}
        for state, counts in self.transitions.items():
            total = sum(counts.values())
            self.probs[state] = {k: v/total for k, v in counts.items()}

    def generate(self, seed, length=96):
        current = seed
        result = []
        for _ in range(length):
            if current in self.probs:
                tokens = list(self.probs[current].keys())
                probs  = list(self.probs[current].values())
                current = random.choices(tokens, probs)[0]
            else:
                current = random.choice(vocab)  # fallback
            result.append(current)
        return result


# Fit Markov chain on training data
mc = MarkovChain()
mc.fit(all_sequences)

# Generate from each method
random_melody = generate_random_baseline(96)
markov_melody = mc.generate(seed=all_sequences[0][0], length=96)
lstm_melody   = melodies[1.0]  # temperature 1.0

print(f'Random baseline: {len(random_melody)} tokens')
print(f'Markov baseline: {len(markov_melody)} tokens')
print(f'LSTM (T=1.0):    {len(lstm_melody)} tokens')

In [ ]:
from scipy.stats import entropy as scipy_entropy

def evaluate_melody(melody_tokens, name=''):
    """Compute all evaluation metrics for a melody."""
    pitches    = [p for p, d in melody_tokens]
    pcs        = [p % 12 for p in pitches]
    intervals  = [abs(pitches[i+1] - pitches[i]) for i in range(len(pitches)-1)]

    # Scale adherence
    in_scale   = sum(1 for pc in pcs if pc in BHAIRAV_PCS)
    scale_pct  = 100 * in_scale / len(pcs)

    # Vadi check: is Ma (pc=5) the most common?
    pc_freq    = Counter(pcs)
    top_note   = pc_freq.most_common(1)[0][0]
    vadi_pct   = 100 * pc_freq.get(5, 0) / len(pcs)
    vadi_rank  = [pc for pc, _ in pc_freq.most_common()].index(5) + 1 if 5 in pc_freq else -1

    # Stepwise motion
    stepwise   = 100 * sum(1 for iv in intervals if iv <= 2) / len(intervals)

    # Pitch entropy (diversity)
    pc_probs   = np.array([pc_freq.get(pc, 0) for pc in sorted(BHAIRAV_PCS)])
    pc_probs   = pc_probs / pc_probs.sum() if pc_probs.sum() > 0 else pc_probs
    pitch_ent  = scipy_entropy(pc_probs, base=2)

    return {
        'name':         name,
        'scale_adh':    scale_pct,
        'vadi_pct':     vadi_pct,
        'vadi_rank':    vadi_rank,
        'stepwise':     stepwise,
        'pitch_entropy': pitch_ent,
        'pc_freq':      pc_freq,
        'intervals':    intervals,
    }


results = [
    evaluate_melody(random_melody, 'Random'),
    evaluate_melody(markov_melody, 'Markov (order-1)'),
    evaluate_melody(lstm_melody,   'LSTM (T=1.0)'),
]

# Also evaluate training data for reference
flat_train = [tok for seq in all_sequences[:200] for tok in seq]
results.append(evaluate_melody(flat_train, 'Training Data'))

# Print comparison table
print(f'{"Method":<20s} {"Scale%":>7s} {"Vadi%":>7s} {"Vadi Rank":>10s} {"Stepwise%":>10s} {"Pitch Ent":>10s}')
print('─' * 68)
for r in results:
    print(f'{r["name"]:<20s} {r["scale_adh"]:>6.1f}% {r["vadi_pct"]:>6.1f}% {r["vadi_rank"]:>8d}     {r["stepwise"]:>7.1f}%  {r["pitch_entropy"]:>8.3f}')

### 3.3 Interpreting the Results

**What to expect from each method:**

- **Random**: 100% scale adherence (we only sample from valid notes), but *no* vadi emphasis, *no* stepwise preference, and high entropy. It sounds like a child randomly poking keys on a scale — technically correct but musically meaningless.

- **Markov (order-1)**: Captures immediate transitions (e.g., Ma often follows Ga) but has no memory beyond one step. Vadi emphasis will be partial. Motion will be somewhat stepwise but occasionally jump wildly because it doesn't know about the broader phrase context.

- **LSTM**: Should score closest to training data across *all* metrics simultaneously — scale adherence, vadi emphasis, stepwise preference, AND reasonable entropy. This is the key result: no single metric proves the LSTM is better; it's the combination that demonstrates it learned the *full grammar*.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
method_names = [r['name'] for r in results]
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#95a5a6']

# 1. Pitch class distribution comparison
ax = axes[0]
pcs_sorted = sorted(BHAIRAV_PCS)
x_pos = np.arange(len(pcs_sorted))
width = 0.2
for i, r in enumerate(results):
    total = sum(r['pc_freq'].values())
    freqs = [r['pc_freq'].get(pc, 0) / total for pc in pcs_sorted]
    ax.bar(x_pos + i * width, freqs, width, label=r['name'], color=colors[i], alpha=0.85)
ax.set_xticks(x_pos + 1.5 * width)
ax.set_xticklabels([NOTE_NAMES[pc] for pc in pcs_sorted], fontsize=8)
ax.set_title('Pitch Class Distribution by Method')
ax.set_ylabel('Proportion'); ax.legend(fontsize=8)

# 2. Interval distribution comparison
ax = axes[1]
bins = range(0, 15)
for i, r in enumerate(results):
    abs_int = [abs(iv) for iv in r['intervals']]
    int_freq = Counter(abs_int)
    total = len(abs_int)
    ax.plot([int_freq.get(b, 0)/total for b in bins], label=r['name'],
            color=colors[i], linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Interval (semitones)'); ax.set_ylabel('Proportion')
ax.set_title('Interval Distribution by Method')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 3. Summary radar/bar chart
ax = axes[2]
metrics = ['Scale %', 'Vadi %', 'Stepwise %']
x_pos = np.arange(len(metrics))
width = 0.2
for i, r in enumerate(results):
    vals = [r['scale_adh'], r['vadi_pct'], r['stepwise']]
    ax.bar(x_pos + i * width, vals, width, label=r['name'], color=colors[i], alpha=0.85)
ax.set_xticks(x_pos + 1.5 * width)
ax.set_xticklabels(metrics)
ax.set_title('Key Metrics Comparison')
ax.set_ylabel('Percentage'); ax.legend(fontsize=8)

plt.suptitle('Evaluation: LSTM vs Baselines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=150)
plt.show()

In [ ]:
# ── Temperature effect analysis ───────────────────────────────────────────────
print('\n── LSTM Temperature Comparison ─────────────────────')
print(f'{"Temp":<8s} {"Scale%":>7s} {"Vadi%":>7s} {"Stepwise%":>10s} {"Entropy":>8s}')
print('─' * 44)
for temp in [0.5, 1.0, 1.5]:
    r = evaluate_melody(melodies[temp], f'T={temp}')
    print(f'{temp:<8.1f} {r["scale_adh"]:>6.1f}% {r["vadi_pct"]:>6.1f}% {r["stepwise"]:>8.1f}%  {r["pitch_entropy"]:>7.3f}')

print('\nInterpretation:')
print('  T=0.5 → conservative, repetitive (low entropy), very safe grammar')
print('  T=1.0 → balanced creativity and correctness')
print('  T=1.5 → adventurous, may occasionally break rules')

---
# Section 4: Related Work

## 4.1 Indian Classical Music Generation

Indian classical music generation is a relatively underexplored area compared to Western music:

- **Suryanarayan et al. (2021)** used LSTMs and Transformers trained on Carnatic music audio to generate raga-specific melodies, finding that LSTMs performed comparably to Transformers on this domain due to the structured nature of ragas. Our synthetic approach bypasses the audio-to-symbolic conversion challenge they faced.

- **The CompMusic project** (Serra et al., MTG Barcelona) built the largest open dataset for Indian classical music (Saraga), focusing on Carnatic and Hindustani traditions with pitch, rhythm, and structural annotations. Their dataset is primarily audio-based; we chose synthetic generation to work directly in the symbolic domain.

- **Sitar Raga Synthesis (various)**: Several small-scale projects have used Markov chains and genetic algorithms for raga generation, but these typically focus on a single aspect (pitch only) without duration modeling.

## 4.2 General Music Generation with LSTMs

- **MelodyRNN** (Google Magenta, 2016): One of the first LSTM-based melody generators, using a similar next-note-prediction approach on a large MIDI corpus. Our architecture is a simplified version of their model, adapted for a single raga rather than a general music distribution.

- **BachBot** (Liang, 2016): An LSTM trained on Bach chorales that achieved near-human performance in a Turing test. Demonstrates that LSTMs can capture complex harmonic grammar. Our vadi/samvadi emphasis analysis is analogous to their analysis of voice-leading rules.

- **Music Transformer** (Huang et al., 2018): Used relative attention to capture long-range structure in piano music. While Transformers are now dominant in music generation, they require significantly more data and compute than our project's scope allows.

## 4.3 Our Contribution

Our work differs from prior approaches in two key ways:
1. **Grammar-based synthetic data**: Rather than collecting recordings, we generate training data from formalized raga rules. This is viable specifically because ragas ARE their rules — unlike jazz or pop where style is emergent.
2. **Explicit raga property evaluation**: We go beyond perplexity to measure whether the model has learned specific musical properties (vadi emphasis, stepwise motion, scale adherence) and compare against baselines.

---
# 5. Export: MIDI → MP3

In [ ]:
def melody_to_midi(tokens, filename, bpm=55):
    """Convert (pitch, duration) token list to MIDI."""
    s = stream.Score()
    part = stream.Part()
    part.append(instrument.Sitar())
    part.append(tempo.MetronomeMark(number=bpm))
    for pitch_midi, dur in tokens:
        n = note.Note(pitch_midi)
        n.quarterLength = dur
        part.append(n)
    s.append(part)
    s.write('midi', fp=filename)
    print(f'MIDI saved: {filename}')


SOUNDFONT = '/usr/share/sounds/sf2/FluidR3_GM.sf2'

def midi_to_mp3(midi_path, mp3_path):
    wav = midi_path.replace('.mid', '.wav')
    subprocess.run(['fluidsynth', '-ni', SOUNDFONT, midi_path, '-F', wav, '-r', '44100'],
                   check=True, capture_output=True)
    subprocess.run(['ffmpeg', '-y', '-i', wav, '-codec:a', 'libmp3lame', '-qscale:a', '2', mp3_path],
                   check=True, capture_output=True)
    os.remove(wav)
    print(f'MP3 saved: {mp3_path}')


# Export LSTM melodies at all temperatures
for temp, mel in melodies.items():
    melody_to_midi(mel, f'bhairav_temp{temp}.mid', bpm=55)
    midi_to_mp3(f'bhairav_temp{temp}.mid', f'bhairav_temp{temp}.mp3')

# Export baselines for comparison
melody_to_midi(random_melody, 'bhairav_random.mid', bpm=55)
midi_to_mp3('bhairav_random.mid', 'bhairav_random.mp3')

melody_to_midi(markov_melody, 'bhairav_markov.mid', bpm=55)
midi_to_mp3('bhairav_markov.mid', 'bhairav_markov.mp3')

print('\nAll files exported.')

## 6. Listen & Compare

In [ ]:
from IPython.display import Audio, display

print('── Random Baseline ──')
display(Audio('bhairav_random.mp3'))

print('\n── Markov Chain (Order-1) ──')
display(Audio('bhairav_markov.mp3'))

for temp in [0.5, 1.0, 1.5]:
    print(f'\n── LSTM Temperature = {temp} ──')
    display(Audio(f'bhairav_temp{temp}.mp3'))

## 7. Save Checkpoint

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab, 'token2idx': token2idx, 'idx2token': idx2token,
    'window_size': WINDOW_SIZE, 'raga': BHAIRAV['name'],
}, 'raga_bhairav_checkpoint.pt')

files = [
    'raga_bhairav_checkpoint.pt', 'training_curves.png', 'eda_plots.png', 'evaluation_plots.png',
    'bhairav_temp0.5.mid',  'bhairav_temp1.0.mid',  'bhairav_temp1.5.mid',
    'bhairav_temp0.5.mp3',  'bhairav_temp1.0.mp3',  'bhairav_temp1.5.mp3',
    'bhairav_random.mp3',   'bhairav_markov.mp3',
]
print('Output files:')
for f in files:
    s = f'✅ {os.path.getsize(f):>10,} bytes' if os.path.exists(f) else '❌ missing'
    print(f'  {s}  {f}')